In [1]:
import numpy as np
import astropy.units as u
import threading as th
import time 
from datetime import datetime
today = int(datetime.today().strftime('%Y%m%d'))
from IPython.display import clear_output
from importlib import reload
import copy
import os
from pathlib import Path

cwd = Path(os.getcwd())

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle, Rectangle

import scoob_llowfsc.llowfsc as llowfsc
import scoob_llowfsc.scoob_interface as scoobi
import scoob_llowfsc.utils as utils
import scoob_llowfsc.telem as telem
from scoob_llowfsc.math_module import xp, xcipy, ensure_np_array
from scoob_llowfsc.imshows import imshow1, imshow2, imshow3

from magpyx.utils import ImageStream
import purepyindi
from purepyindi import INDIClient
import purepyindi2
from purepyindi2 import IndiClient

client0 = INDIClient('localhost', 7624)
client0.start()
client = IndiClient()
client.connect()
client.get_properties()

def restart_clients():
    client0 = INDIClient('localhost', 7624)
    client0.start()

    client = IndiClient()
    client.connect()
    client.get_properties()

wavelength = 633e-9


/opt/conda/envs/km310env/lib/python3.10/site-packages/cupyx/jit/_interface.py:173: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


Could not import deepdish!


In [ ]:
reload(scoobi)
xc, yc = (4600, 3400)
npsf = 256
scoobi.set_zwo_roi(xc, yc, npsf, client0)

In [7]:
dm_channel = 'dm00disp01'
cam_channel = 'camsci'

dm_stream = ImageStream(dm_channel)
cam_stream = ImageStream(cam_channel)

In [ ]:
def test_fun(
        dm_stream,
        cam_stream,
        rms=5e-9, 
        plot=False,
    ):

    frame = cam_stream.grab_latest()
    dm_command = rms*np.random.randn(dm_stream.shape)
    dm_stream.write(1e6 * dm_command)

    if plot:
        imshow2(frame, dm_command)


In [10]:
reload(llowfsc)
reload(utils)

test_freq = 1
test_interval = 1/test_freq
print(test_interval)

args = [
    dm_stream,
    cam_stream,
]

kwargs = {
    'rms':10e-9,
    'plot':1,
}

test_process = llowfsc.Process(
    test_interval, 
    llowfsc.single_iteration, 
    args, # the args
    kwargs, # the kwargs
)

1.0


In [14]:
reload(telem)
exp_parent_path = Path(f'{cwd}/data/threading-tests')
exp_dir = f'freq-{test_freq:.2f}'
exp_path = exp_parent_path / exp_dir

dm_data_path = exp_path / dm_channel
cam_data_path = exp_path / cam_channel

telem.make_dir(exp_parent_path)
telem.make_dir(exp_path)
telem.make_dir(dm_data_path)
telem.make_dir(cam_data_path)

Directory '/home/kianmilani/Projects/scoob-llowfsc/notebooks/data/threading-tests' already exists.
Directory '/home/kianmilani/Projects/scoob-llowfsc/notebooks/data/threading-tests/freq-1.00' already exists.
Directory '/home/kianmilani/Projects/scoob-llowfsc/notebooks/data/threading-tests/freq-1.00/dm00disp01' already exists.
Directory '/home/kianmilani/Projects/scoob-llowfsc/notebooks/data/threading-tests/freq-1.00/camsci' already exists.


In [15]:
reload(telem)
telem.delete_files(cam_data_path/'*')
telem.delete_files(dm_data_path/'*')
telem.delete_files(telem.dm01_path/'*')
telem.delete_files(telem.camsci_path/'*')

In [ ]:
telem.toggle(1, cam_channel, client0)
telem.toggle(1, dm_channel, client0)
test_process.start()

time.sleep(3)

test_process.cancel()
dm_stream.write(np.zeros(dm_stream.shape))
telem.toggle(0, cam_channel, client0)
telem.toggle(0, dm_channel, client0)

In [ ]:
telem.move_files(telem.cam_path, cam_data_path)
telem.move_files(telem.dm_path, dm_data_path)

In [ ]:
reload(telem)
telem.unpack_data(cam_data_path, cam_data_path)
telem.unpack_data(dm_data_path, dm_data_path)

In [ ]:
reload(telem)

cam_data_fnames = telem.get_fnames(cam_data_path/'campupil*20250212*.fits')
dm_data_fnames = telem.get_fnames(dm_data_path/'dm00disp01*20250212*.fits')

dm_data_fnames[0], cam_data_fnames[0]

In [ ]:
from astropy.io import fits

dm_commands = []
dm_times = []
for fname in dm_data_fnames:
    dm_commands.append(fits.getdata(fname))
    t_hr = float(fname.split("_")[1][8:10])
    t_min = float(fname.split("_")[1][10:12])
    t_sec = float(fname.split("_")[1][12:-5])/1e9
    dm_times.append( 3600*t_hr + 60*t_min + t_sec )
dm_commands = np.array(dm_commands) 
dm_times = np.array(dm_times)
dm_start = dm_times[0]
rel_dm_times = dm_times - dm_start

frames = []
cam_times = []
for fname in cam_data_fnames:
    frames.append(fits.getdata(fname))
    t_hr = float(fname.split("_")[1][8:10])
    t_min = float(fname.split("_")[1][10:12])
    t_sec = float(fname.split("_")[1][12:-5])/1e9
    cam_times.append( 3600*t_hr + 60*t_min + t_sec )
frames = np.array(frames) 
cam_times = np.array(cam_times)
cam_start = cam_times[0]
rel_cam_times = cam_times - dm_start

print(dm_start) 
print(cam_start)
print(dm_start - cam_start)

dm_time_steps = (rel_dm_times[1:] - rel_dm_times[:-1])
print(test_interval, np.mean(dm_time_steps))
print(test_freq, 1/np.mean(dm_time_steps))

plt.figure(figsize=(12,5))
plt.subplot(121)
plt.plot(rel_dm_times)
plt.ylabel('Relative times for DM commands [s]')
plt.xlabel('command count')

plt.subplot(122)
plt.plot(rel_cam_times)
plt.ylabel('Relative Times for camera data [s]')
plt.xlabel('frame count')

plt.figure()
plt.plot(dm_time_steps)